# This code takes all of the COUNTER 5 and COUNTER 5.1 data available and combines it into the Big_Usage_Database_Updated for use in our Usage PowerBI report and other usage-related projects.
## Individual usage files, along with the database and files created as part of this code, are currently stored on SharePoint. For security reasons, any usage needed will need to be either downloaded to a local file and/or be saved locally before uploading to SharePoint.


# Here are some notes to help you best use this code:
## This code is designed to load one usage file type at a time for consistency. If your folder has multiple types of COUNTER usage, please split them up into multiple folders by file type before uploading to this document. Any number of files of a single usage type can be run at once using this code.
## If you are loading COUNTER 5 and 5.1 usage, and/or adding usage gathered via SUSHI/the COUNTER API Python code, separate those into separate folders named after the respective format.
## COUNTER 4 or older, along with Non-COUNTER usage, will not be included in this database.

### Load Packages 

In [1]:
import numpy as np
import pandas as pd
import os
from os import listdir
from os.path import isfile, join
import shutil
from pprint import pprint
import csv
import sqlite3
from sqlite3 import Error
from pathlib import Path
import ast
import warnings
import xlsxwriter
import janitor
from datetime import date
from dateutil.relativedelta import relativedelta
from datetime import datetime

warnings.filterwarnings("ignore") #This is here to improve performance time. However, if you are repairing, improving, or otherwise testing this code, it is recommended to comment this out.

# Load Usage from a Local Folder

### Copy the full path to the folder where the usage files are stored on your computer, and place the path in the quotes below. 

The easiest method to do this is by going into your files, right-clicking on the folder, and selecting "Copy Address as Text" or "Copy as Path" (phrasing may differ on Mac).

## Make sure the folderPath starts with an 'r' and that the path is in full quotation marks.

In [9]:
folderPath = r"C:\Users\aharrima\OneDrive - University of Tennessee\Documents\Usage Database Update\Single File Holding Area\TRB3\VolBooks Holding Area"

directory = os.chdir(folderPath)

## Establish Connection to the SQLite Database

### Just like the usage folder, copy the full path to where the database is stored on your computer and place that path in the quotes below.

In [10]:
#establish connection to Big Usage Database and create cursor object
conn = sqlite3.connect(
    r"C:\Users\aharrima\OneDrive - University of Tennessee\Documents\Big_Usage_Database_Updated.db"
)
cur = conn.cursor()

### If new tables are needed in the database, use the code below to add them.

## WARNING: Running this on a table already in the database will wipe all data from it!!! Leave this commented out if it is not needed.

In [11]:
default = pd.DataFrame()

#default.to_sql('PR_Archive', conn, if_exists = 'replace')

### List All Files in the Folder

In [12]:
files = [f for f in listdir(folderPath) if isfile(join(folderPath, f))]

len(files)

7

### Show all File Extension Types in the Folder

#### This code will only work on .csv, .tsv, or .xlsx files. All other files will be skipped. If your folder contains other file types, please convert them to one of these types before continuing.

In [13]:
extensions = {os.path.splitext(file)[1] for file in os.listdir(folderPath) if os.path.isfile(os.path.join(folderPath, file))}

extensions

{'.xlsx'}

## List the COUNTER Usage Type Here

#### This will be referenced later in the code for ease of use. Make sure to include the underscore if you are working with a specific report, such as the TR_B3 or the PR_P1.

In [14]:
fileType = 'TR_B3'

# Load and Combine Files from the Folder

### NOTE: This code works best when the file names follow the standard format: VendorName-ReportType-FirstMonth-LastMonth. If your files do not follow this format, then either change their names to this format or edit the code to best suit your file names.

In [15]:
newUsage = pd.DataFrame()

singleFile = pd.DataFrame()

# To account for COUNTER 5.1's new file header type, which adds an additional row, and the Python header type, which is no header
if 'COUNTER 5.1' in folderPath:
    skipRowNum = 14
elif 'From Python' in folderPath:
    skipRowNum = 0
elif 'VolBooks Holding Area' in folderPath:
    skipRowNum = 0
else:
    skipRowNum = 13

## Column Name Changes Dictionary

### This dictionary changes the column names into a consistent format, as various files can have various ways of phrasing important information (especially months).
### This will need to be updated every six months to include new month names, along with any time rogue names may appear from a vendor's usage file.

In [16]:
columnRenameDict = {
    #Jul-Dec 2026
    '07-2026':'Jul-2026',
    '08-2026':'Aug-2026',
    '09-2026':'Sep-2026',
    '10-2026':'Oct-2026',
    '11-2026':'Nov-2026',
    '12-2026':'Dec-2026',
    '2026-07':'Jul-2026',
    '2026-08':'Aug-2026',
    '2026-09':'Sep-2026',
    '2026-10':'Oct-2026',
    '2026-11':'Nov-2026',
    '2026-12':'Dec-2026',
    '2026-07-01 00:00:00':'Jul-2026',
    '2026-08-01 00:00:00':'Aug-2026',
    '2026-09-01 00:00:00':'Sep-2026',
    '2026-10-01 00:00:00':'Oct-2026',
    '2026-11-01 00:00:00':'Nov-2026',
    '2026-12-01 00:00:00':'Dec-2026',
    '2026-07-26 00:00:00':'Jul-2026',
    '2026-08-26 00:00:00':'Aug-2026',
    '2026-09-26 00:00:00':'Sep-2026',
    '2026-10-26 00:00:00':'Oct-2026',
    '2026-11-26 00:00:00':'Nov-2026',
    '2026-12-26 00:00:00':'Dec-2026',
    '2026-07-26 00:00:00':'Jul-2026',
    '2026-08-26 00:00:00':'Aug-2026',
    '2026-09-26 00:00:00':'Sep-2026',
    '2026-10-26 00:00:00':'Oct-2026',
    '2026-11-26 00:00:00':'Nov-2026',
    '2026-12-26 00:00:00':'Dec-2026',
    '26-Jul':'Jul-2026',
    '26-Aug':'Aug-2026',
    '26-Sep':'Sep-2026',
    '26-Oct':'Oct-2026',
    '26-Nov':'Nov-2026',
    '26-Dec':'Dec-2026',
    'Jul-26':'Jul-2026',
    'Aug-26':'Aug-2026',
    'Sep-26':'Sep-2026',
    'Oct-26':'Oct-2026',
    'Nov-26':'Nov-2026',
    'Dec-26':'Dec-2026',
    
    #Jan-Jun 2026
    '01-2026':'Jan-2026',
    '02-2026':'Feb-2026',
    '03-2026':'Mar-2026',
    '04-2026':'Apr-2026',
    '05-2026':'May-2026',
    '06-2026':'Jun-2026',
    '2026-01':'Jan-2026',
    '2026-02':'Feb-2026',
    '2026-03':'Mar-2026',
    '2026-04':'Apr-2026',
    '2026-05':'May-2026',
    '2026-06':'Jun-2026',
    '2026-01-01 00:00:00':'Jan-2026',
    '2026-02-01 00:00:00':'Feb-2026',
    '2026-03-01 00:00:00':'Mar-2026',
    '2026-04-01 00:00:00':'Apr-2026',
    '2026-05-01 00:00:00':'May-2026',
    '2026-06-01 00:00:00':'Jun-2026',
    '2026-02-11T06:56:39Z':'Feb-2026',
    '2026-01-26 00:00:00':'Jan-2026',
    '2026-02-26 00:00:00':'Feb-2026',
    '2026-03-26 00:00:00':'Mar-2026',
    '2026-04-26 00:00:00':'Apr-2026',
    '2026-05-26 00:00:00':'May-2026',
    '2026-06-26 00:00:00':'Jun-2026',
    '26-Jan':'Jan-2026',
    '26-Feb':'Feb-2026',
    '26-Mar':'Mar-2026',
    '26-Apr':'Apr-2026',
    '26-May':'May-2026',
    '26-Jun':'Jun-2026',
    'Jan-26':'Jan-2026',
    'Feb-26':'Feb-2026',
    'Mar-26':'Mar-2026',
    'Apr-26':'Apr-2026',
    'May-26':'May-2026',
    'Jun-26':'Jun-2026',

    
    #Jul-Dec 2025
    '07-2025':'Jul-2025',
    '08-2025':'Aug-2025',
    '09-2025':'Sep-2025',
    '10-2025':'Oct-2025',
    '11-2025':'Nov-2025',
    '12-2025':'Dec-2025',
    '2025-07':'Jul-2025',
    '2025-08':'Aug-2025',
    '2025-09':'Sep-2025',
    '2025-10':'Oct-2025',
    '2025-11':'Nov-2025',
    '2025-12':'Dec-2025',
    '2025-07-01 00:00:00':'Jul-2025',
    '2025-08-01 00:00:00':'Aug-2025',
    '2025-09-01 00:00:00':'Sep-2025',
    '2025-10-01 00:00:00':'Oct-2025',
    '2025-11-01 00:00:00':'Nov-2025',
    '2025-12-01 00:00:00':'Dec-2025',
    '2025-07-25 00:00:00':'Jul-2025',
    '2025-08-25 00:00:00':'Aug-2025',
    '2025-09-25 00:00:00':'Sep-2025',
    '2025-10-25 00:00:00':'Oct-2025',
    '2025-11-25 00:00:00':'Nov-2025',
    '2025-12-25 00:00:00':'Dec-2025',
    '2026-07-25 00:00:00':'Jul-2025',
    '2026-08-25 00:00:00':'Aug-2025',
    '2026-09-25 00:00:00':'Sep-2025',
    '2026-10-25 00:00:00':'Oct-2025',
    '2026-11-25 00:00:00':'Nov-2025',
    '2026-12-25 00:00:00':'Dec-2025',
    '25-Jul':'Jul-2025',
    '25-Aug':'Aug-2025',
    '25-Sep':'Sep-2025',
    '25-Oct':'Oct-2025',
    '25-Nov':'Nov-2025',
    '25-Dec':'Dec-2025',
    'Jul-25':'Jul-2025',
    'Aug-25':'Aug-2025',
    'Sep-25':'Sep-2025',
    'Oct-25':'Oct-2025',
    'Nov-25':'Nov-2025',
    'Dec-25':'Dec-2025',

    #Jan-Jun 2025
    '01-2025':'Jan-2025',
    '02-2025':'Feb-2025',
    '03-2025':'Mar-2025',
    '04-2025':'Apr-2025',
    '05-2025':'May-2025',
    '06-2025':'Jun-2025',
    '2025-01':'Jan-2025',
    '2025-02':'Feb-2025',
    '2025-03':'Mar-2025',
    '2025-04':'Apr-2025',
    '2025-05':'May-2025',
    '2025-06':'Jun-2025',
    '2025-01-01 00:00:00':'Jan-2025',
    '2025-02-01 00:00:00':'Feb-2025',
    '2025-03-01 00:00:00':'Mar-2025',
    '2025-04-01 00:00:00':'Apr-2025',
    '2025-05-01 00:00:00':'May-2025',
    '2025-06-01 00:00:00':'Jun-2025',
    '2025-02-11T06:56:39Z':'Feb-2025',
    '2025-01-25 00:00:00':'Jan-2025',
    '2025-02-25 00:00:00':'Feb-2025',
    '2025-03-25 00:00:00':'Mar-2025',
    '2025-04-25 00:00:00':'Apr-2025',
    '2025-05-25 00:00:00':'May-2025',
    '2025-06-25 00:00:00':'Jun-2025',
    '25-Jan':'Jan-2025',
    '25-Feb':'Feb-2025',
    '25-Mar':'Mar-2025',
    '25-Apr':'Apr-2025',
    '25-May':'May-2025',
    '25-Jun':'Jun-2025',
    'Jan-25':'Jan-2025',
    'Feb-25':'Feb-2025',
    'Mar-25':'Mar-2025',
    'Apr-25':'Apr-2025',
    'May-25':'May-2025',
    'Jun-25':'Jun-2025',

    #Jul-Dec 2024
    '07-2024':'Jul-2024',
    '08-2024':'Aug-2024',
    '09-2024':'Sep-2024',
    '10-2024':'Oct-2024',
    '11-2024':'Nov-2024',
    '12-2024':'Dec-2024',
    '2024-07':'Jul-2024',
    '2024-08':'Aug-2024',
    '2024-09':'Sep-2024',
    '2024-10':'Oct-2024',
    '2024-11':'Nov-2024',
    '2024-12':'Dec-2024',
    '2024-07-01 00:00:00':'Jul-2024',
    '2024-08-01 00:00:00':'Aug-2024',
    '2024-09-01 00:00:00':'Sep-2024',
    '2024-10-01 00:00:00':'Oct-2024',
    '2024-11-01 00:00:00':'Nov-2024',
    '2024-12-01 00:00:00':'Dec-2024',
    '2025-07-24 00:00:00':'Jul-2024',
    '2025-08-24 00:00:00':'Aug-2024',
    '2025-09-24 00:00:00':'Sep-2024',
    '2025-10-24 00:00:00':'Oct-2024',
    '2025-11-24 00:00:00':'Nov-2024',
    '2025-12-24 00:00:00':'Dec-2024',
    '24-Jul':'Jul-2024',
    '24-Aug':'Aug-2024',
    '24-Sep':'Sep-2024',
    '24-Oct':'Oct-2024',
    '24-Nov':'Nov-2024',
    '24-Dec':'Dec-2024',
    'Jul-24':'Jul-2024',
    'Aug-24':'Aug-2024',
    'Sep-24':'Sep-2024',
    'Oct-24':'Oct-2024',
    'Nov-24':'Nov-2024',
    'Dec-24':'Dec-2024',
    'Sept-2024':'Sep-2024',

    #Jan-Jun 2024
    '01-2024':'Jan-2024',
    '02-2024':'Feb-2024',
    '03-2024':'Mar-2024',
    '04-2024':'Apr-2024',
    '05-2024':'May-2024',
    '06-2024':'Jun-2024',
    '2024-01':'Jan-2024',
    '2024-02':'Feb-2024',
    '2024-03':'Mar-2024',
    '2024-04':'Apr-2024',
    '2024-05':'May-2024',
    '2024-06':'Jun-2024',
    '2024-01-01 00:00:00':'Jan-2024',
    '2024-02-01 00:00:00':'Feb-2024',
    '2024-03-01 00:00:00':'Mar-2024',
    '2024-04-01 00:00:00':'Apr-2024',
    '2024-05-01 00:00:00':'May-2024',
    '2024-06-01 00:00:00':'Jun-2024',
    '2025-01-24 00:00:00':'Jan-2024',
    '2025-02-24 00:00:00':'Feb-2024',
    '2025-03-24 00:00:00':'Mar-2024',
    '2025-04-24 00:00:00':'Apr-2024',
    '2025-05-24 00:00:00':'May-2024',
    '2025-06-24 00:00:00':'Jun-2024',
    '24-Jan':'Jan-2024',
    '24-Feb':'Feb-2024',
    '24-Mar':'Mar-2024',
    '24-Apr':'Apr-2024',
    '24-May':'May-2024',
    '24-Jun':'Jun-2024',
    'Jan-24':'Jan-2024',
    'Feb-24':'Feb-2024',
    'Mar-24':'Mar-2024',
    'Apr-24':'Apr-2024',
    'May-24':'May-2024',
    'Jun-24':'Jun-2024',

    #Jul-Dec 2023
    '2023-07':'Jul-2023',
    '2023-08':'Aug-2023',
    '2023-09':'Sep-2023',
    '2023-10':'Oct-2023',
    '2023-11':'Nov-2023',
    '2023-12':'Dec-2023',
    '2023-07-01 00:00:00':'Jul-2023',
    '2023-08-01 00:00:00':'Aug-2023',
    '2023-09-01 00:00:00':'Sep-2023',
    '2023-10-01 00:00:00':'Oct-2023',
    '2023-11-01 00:00:00':'Nov-2023',
    '2023-12-01 00:00:00':'Dec-2023',
    '2023-07-22 00:00:00':'Jul-2023',
    '2023-08-22 00:00:00':'Aug-2023',
    '2023-09-22 00:00:00':'Sep-2023',
    '2023-10-22 00:00:00':'Oct-2023',
    '2023-11-22 00:00:00':'Nov-2023',
    '2023-12-22 00:00:00':'Dec-2023',
    '2025-07-23 00:00:00':'Jul-2023',
    '2025-08-23 00:00:00':'Aug-2023',
    '2025-09-23 00:00:00':'Sep-2023',
    '2025-10-23 00:00:00':'Oct-2023',
    '2025-11-23 00:00:00':'Nov-2023',
    '2025-12-23 00:00:00':'Dec-2023',
    '23-Jul':'Jul-2023',
    '23-Aug':'Aug-2023',
    '23-Sep':'Sep-2023',
    '23-Oct':'Oct-2023',
    '23-Nov':'Nov-2023',
    '23-Dec':'Dec-2023',
    'Jul-23':'Jul-2023',
    'Aug-23':'Aug-2023',
    'Sep-23':'Sep-2023',
    'Oct-23':'Oct-2023',
    'Nov-23':'Nov-2023',
    'Dec-23':'Dec-2023',
    'Sept-2023':'Sep-2023',

    #Jan-Jun 2023
    '2023-01':'Jan-2023',
    '2023-02':'Feb-2023',
    '2023-03':'Mar-2023',
    '2023-04':'Apr-2023',
    '2023-05':'May-2023',
    '2023-06':'Jun-2023',
    '2023-01-01 00:00:00':'Jan-2023',
    '2023-02-01 00:00:00':'Feb-2023',
    '2023-03-01 00:00:00':'Mar-2023',
    '2023-04-01 00:00:00':'Apr-2023',
    '2023-05-01 00:00:00':'May-2023',
    '2023-06-01 00:00:00':'Jun-2023',
    '2025-01-23 00:00:00':'Jan-2023',
    '2025-02-23 00:00:00':'Feb-2023',
    '2025-03-23 00:00:00':'Mar-2023',
    '2025-04-23 00:00:00':'Apr-2023',
    '2025-05-23 00:00:00':'May-2023',
    '2025-06-23 00:00:00':'Jun-2023',
    '23-Jan':'Jan-2023',
    '23-Feb':'Feb-2023',
    '23-Mar':'Mar-2023',
    '23-Apr':'Apr-2023',
    '23-May':'May-2023',
    '23-Jun':'Jun-2023',
    'Jan-23':'Jan-2023',
    'Feb-23':'Feb-2023',
    'Mar-23':'Mar-2023',
    'Apr-23':'Apr-2023',
    'May-23':'May-2023',
    'Jun-23':'Jun-2023',

    #Jul-Dec 2022
    '2022-07':'Jul-2022',
    '2022-08':'Aug-2022',
    '2022-09':'Sep-2022',
    '2022-10':'Oct-2022',
    '2022-11':'Nov-2022',
    '2022-12':'Dec-2022',
    '2022-07-01 00:00:00':'Jul-2022',
    '2022-08-01 00:00:00':'Aug-2022',
    '2022-09-01 00:00:00':'Sep-2022',
    '2022-10-01 00:00:00':'Oct-2022',
    '2022-11-01 00:00:00':'Nov-2022',
    '2022-12-01 00:00:00':'Dec-2022',
    '22-Jul':'Jul-2022',
    '22-Aug':'Aug-2022',
    '22-Sep':'Sep-2022',
    '22-Oct':'Oct-2022',
    '22-Nov':'Nov-2022',
    '22-Dec':'Dec-2022',
    'Jul-22':'Jul-2022',
    'Aug-22':'Aug-2022',
    'Sep-22':'Sep-2022',
    'Oct-22':'Oct-2022',
    'Nov-22':'Nov-2022',
    'Dec-22':'Dec-2022',

    #Jan-Jun 2022
    '2022-01':'Jan-2022',
    '2022-02':'Feb-2022',
    '2022-03':'Mar-2022',
    '2022-04':'Apr-2022',
    '2022-05':'May-2022',
    '2022-06':'Jun-2022',
    '2022-01-01 00:00:00':'Jan-2022',
    '2022-02-01 00:00:00':'Feb-2022',
    '2022-03-01 00:00:00':'Mar-2022',
    '2022-04-01 00:00:00':'Apr-2022',
    '2022-05-01 00:00:00':'May-2022',
    '2022-06-01 00:00:00':'Jun-2022',
    '2022-01-22 00:00:00':'Jan-2022',
    '2022-02-22 00:00:00':'Feb-2022',
    '2022-03-22 00:00:00':'Mar-2022',
    '2022-04-22 00:00:00':'Apr-2022',
    '2022-05-22 00:00:00':'May-2022',
    '2022-06-22 00:00:00':'Jun-2022',
    '22-Jan':'Jan-2022',
    '22-Feb':'Feb-2022',
    '22-Mar':'Mar-2022',
    '22-Apr':'Apr-2022',
    '22-May':'May-2022',
    '22-Jun':'Jun-2022',
    'Jan-22':'Jan-2022',
    'Feb-22':'Feb-2022',
    'Mar-22':'Mar-2022',
    'Apr-22':'Apr-2022',
    'May-22':'May-2022',
    'Jun-22':'Jun-2022',

    #Jul-Dec 2021
    '2021-07':'Jul-2021',
    '2021-08':'Aug-2021',
    '2021-09':'Sep-2021',
    '2021-10':'Oct-2021',
    '2021-11':'Nov-2021',
    '2021-12':'Dec-2021',
    '2021-07-01 00:00:00':'Jul-2021',
    '2021-08-01 00:00:00':'Aug-2021',
    '2021-09-01 00:00:00':'Sep-2021',
    '2021-10-01 00:00:00':'Oct-2021',
    '2021-11-01 00:00:00':'Nov-2021',
    '2021-12-01 00:00:00':'Dec-2021',
    '2022-07-21 00:00:00':'Jul-2021',
    '2022-08-21 00:00:00':'Aug-2021',
    '2022-09-21 00:00:00':'Sep-2021',
    '2022-10-21 00:00:00':'Oct-2021',
    '2022-11-21 00:00:00':'Nov-2021',
    '2022-12-21 00:00:00':'Dec-2021',
    '21-Jul':'Jul-2021',
    '21-Aug':'Aug-2021',
    '21-Sep':'Sep-2021',
    '21-Oct':'Oct-2021',
    '21-Nov':'Nov-2021',
    '21-Dec':'Dec-2021',
    'Jul-21':'Jul-2021',
    'Aug-21':'Aug-2021',
    'Sep-21':'Sep-2021',
    'Oct-21':'Oct-2021',
    'Nov-21':'Nov-2021',
    'Dec-21':'Dec-2021',


    #Jan-Jun 2021
    '2021-01':'Jan-2021',
    '2021-02':'Feb-2021',
    '2021-03':'Mar-2021',
    '2021-04':'Apr-2021',
    '2021-05':'May-2021',
    '2021-06':'Jun-2021',
    '2021-01-01 00:00:00':'Jan-2021',
    '2021-02-01 00:00:00':'Feb-2021',
    '2021-03-01 00:00:00':'Mar-2021',
    '2021-04-01 00:00:00':'Apr-2021',
    '2021-05-01 00:00:00':'May-2021',
    '2021-06-01 00:00:00':'Jun-2021',
    '21-Jan':'Jan-2021',
    '21-Feb':'Feb-2021',
    '21-Mar':'Mar-2021',
    '21-Apr':'Apr-2021',
    '21-May':'May-2021',
    '21-Jun':'Jun-2021',
    'Jan-21':'Jan-2021',
    'Feb-21':'Feb-2021',
    'Mar-21':'Mar-2021',
    'Apr-21':'Apr-2021',
    'May-21':'May-2021',
    'Jun-21':'Jun-2021',

    #General Terms
    'DataType':'Data_Type', 
    'MetricType':'Metric_Type',
    'AccessType':'Access_Type',
    
}

### Load Data

In [17]:
for file in range(0,len(files)):
    if file % 50 == 0: 
        print(file) # Used as a timer for loading many files

    # Get Vendor Name
    name = files[file]
    if name == 'desktop.ini': #This may appear as a junk file in your folder after running this code. It is not important, so can be skipped.
        continue
        
    if '2022-Jul' in name:
        vendor = str(name.split('_')[0]) #Catches the different format of July-December 2022 file usage
    else:
        vendor = str(name.split('-')[0]) #For files in the standard format shown above

    # Load the Usage File
    if name[-4:] != 'xlsx' and name[-3:] != 'tsv' and name[-3:] != 'csv':
        print(r'File not in .xlsx, .csv, or .tsv format - '+{vendor})
        continue
    elif name[-4:] == 'xlsx':
        singleFile = pd.read_excel(name, skiprows = skipRowNum)
    elif name[-3:] == 'tsv':
        singleFile = pd.read_csv(name, sep = '\t', skiprows = skipRowNum, encoding = 'latin-1')
    else:
        singleFile = pd.read_csv(name, skiprows = skipRowNum, encoding = 'latin-1')

    # Check for files with no data
    if singleFile.shape[0] == 0:
        print(r'File has no usage data: ' + name)
        continue
    singleFile['Vendor'] = vendor
    
    # Normalize Date Column Names
    singleFile.columns = singleFile.columns.map(str)
    singleFile.rename(columns = columnRenameDict, inplace = True)

    # Combine the single files into one large file
    if newUsage.shape[0] == 0:
        newUsage = singleFile
        continue
    newUsage = pd.concat([newUsage, singleFile], ignore_index = True)


print('All files loaded')

0
All files loaded


In [43]:
#singleFile

In [49]:
#newUsageA = pd.read_sql_query("SELECT * from TR_J3_C5_Load_All;", conn)
#newUsageB = pd.read_sql_query("SELECT * from TR_J3_C51_Load_All;", conn)
#newUsageC = pd.read_sql_query("SELECT * from TR_J3_Python_Load_All;", conn)
#newUsageD = pd.read_sql_query("SELECT * from TR_C5_Raw_750_End;", conn)

### Combining Data

If you are loading more than one type of usage, use the cells below to store each piece so they can all be added to the database in one run. If you are only running one header type, then you can skip to the "Clean Up" section.

In [31]:
#newUsageA = newUsage #The main COUNTER 5.0 setup
newUsageB = newUsage #The main COUNTER 5.1 setup
#newUsageC = newUsage #The Python usage

In [32]:
newUsage = pd.concat([newUsageA, newUsageB, newUsageC], ignore_index = True)

NameError: name 'newUsageA' is not defined

In [18]:
newUsage.shape

(53442, 17)

In [19]:
newUsage

,Title,Publisher,Platform,Access_Type,Metric_Type,Vendor,Jul-2026,Aug-2026,Reporting_Period_Total,Publisher_ID,DOI,Proprietary_ID,ISBN,Print_ISSN,Online_ISSN,URI,YOP
0,(Out) on the Road: The Radical Joy of Queer Tr...,Ig Publishing,EBSCOhost,Controlled,Total_Item_Investigations,EBSCO,0.0,40.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,(Out) on the Road: The Radical Joy of Queer Tr...,Ig Publishing,EBSCOhost,Controlled,Total_Item_Requests,EBSCO,0.0,38.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(Out) on the Road: The Radical Joy of Queer Tr...,Ig Publishing,EBSCOhost,Controlled,Unique_Item_Investigations,EBSCO,0.0,38.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,(Out) on the Road: The Radical Joy of Queer Tr...,Ig Publishing,EBSCOhost,Controlled,Unique_Item_Requests,EBSCO,0.0,38.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,(Out) on the Road: The Radical Joy of Queer Tr...,Ig Publishing,EBSCOhost,Controlled,Unique_Title_Investigations,EBSCO,0.0,2.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53437,eMagRes,Wiley,Wiley Online Library,Controlled,Total_Item_Requests,Wiley,7.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53438,eMagRes,Wiley,Wiley Online Library,Controlled,Unique_Item_Investigations,Wiley,3.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53439,eMagRes,Wiley,Wiley Online Library,Controlled,Unique_Item_Requests,Wiley,3.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53440,eMagRes,Wiley,Wiley Online Library,Controlled,Unique_Title_Investigations,Wiley,3.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Saving and Loading from the Database Directly

If the usage you are working with is already in the database in its raw format, you can load that data using this cell instead. 
Similarly, if you are loading a large amount of data, you can save it as a table in the database here to save loading time in the future. Skip this cell if neither are needed.

In [20]:
#newUsage.to_sql('TR_B3_Load_All', conn, if_exists = 'replace')
#newUsage = pd.read_sql_query("SELECT * from TR_C51_Load_All;", conn)

# Clean Up the New Usage File

## This section will clean up the usage file and prepare it for addition to the database.

### Remove Unnamed Columns

Sometimes Python adds Unnamed columns to the end of the file; this will remove those columns.

In [21]:
newUsage2 = newUsage.loc[:, ~newUsage.columns.str.contains('^Unnamed')]

newUsage2.rename(columns={'ï»¿Title': 'Title',
                          '="Jul-2021"':'Jul-2021',
                          '="Aug-2021"':'Aug-2021',
                          '="Sep-2021"':'Sep-2021',
                          '="Oct-2021"':'Oct-2021',
                          '="Nov-2021"':'Nov-2021',
                          '="Dec-2021"':'Dec-2021',
                          '="Jan-2022"':'Jan-2022',
                          '="Feb-2022"':'Feb-2022',
                          '="Mar-2022"':'Mar-2022',
                          '="Apr-2022"':'Apr-2022',
                          '="May-2022"':'May-2022',
                          '="Jun-2022"':'Jun-2022',
                          '="Jul-2022"':'Jul-2022',
                          '="Aug-2022"':'Aug-2022',
                          '="Sep-2022"':'Sep-2022',
                          '="Oct-2022"':'Oct-2022',
                          '="Nov-2022"':'Nov-2022',
                          '="Dec-2022"':'Dec-2022',
                          '="Jan-2023"':'Jan-2023',
                          '="Feb-2023"':'Feb-2023',
                          '="Mar-2023"':'Mar-2023',
                          '="Apr-2023"':'Apr-2023',
                          '="May-2023"':'May-2023',
                          '="Jun-2023"':'Jun-2023',
                          '="Jul-2023"':'Jul-2023',
                          '="Aug-2023"':'Aug-2023',
                          '="Sep-2023"':'Sep-2023',
                          '="Oct-2023"':'Oct-2023',
                          '="Nov-2023"':'Nov-2023',
                          '="Dec-2023"':'Dec-2023',
                          '="Jan-2024"':'Jan-2024',
                          '="Feb-2024"':'Feb-2024',
                          '="Mar-2024"':'Mar-2024',
                          '="Apr-2024"':'Apr-2024',
                          '="May-2024"':'May-2024',
                          '="Jun-2024"':'Jun-2024',
                          '="Jul-2024"':'Jul-2024',
                          '="Aug-2024"':'Aug-2024',
                          '="Sep-2024"':'Sep-2024',
                          '="Oct-2024"':'Oct-2024',
                          '="Nov-2024"':'Nov-2024',
                          '="Dec-2024"':'Dec-2024',
                          '="Jan-2025"':'Jan-2025',
                          '="Feb-2025"':'Feb-2025',
                          '="Mar-2025"':'Mar-2025',
                          '="Apr-2025"':'Apr-2025',
                          '="May-2025"':'May-2025',
                          '="Jun-2025"':'Jun-2025',
                          '="Jul-2025"':'Jul-2025',
                          '="Aug-2025"':'Aug-2025',
                          '="Sep-2025"':'Sep-2025',
                          '="Oct-2025"':'Oct-2025',
                          '="Nov-2025"':'Nov-2025',
                          '="Dec-2025"':'Dec-2025',
                          '="Jan-2026"':'Jan-2026',
                          '="Feb-2026"':'Feb-2026',
                          '="Mar-2026"':'Mar-2026',
                          '="Apr-2026"':'Apr-2026',
                          '="May-2026"':'May-2026',
                          '="Jun-2026"':'Jun-2026',}, inplace=True)

newUsage2

,Title,Publisher,Platform,Access_Type,Metric_Type,Vendor,Jul-2026,Aug-2026,Reporting_Period_Total,Publisher_ID,DOI,Proprietary_ID,ISBN,Print_ISSN,Online_ISSN,URI,YOP
0,(Out) on the Road: The Radical Joy of Queer Tr...,Ig Publishing,EBSCOhost,Controlled,Total_Item_Investigations,EBSCO,0.0,40.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,(Out) on the Road: The Radical Joy of Queer Tr...,Ig Publishing,EBSCOhost,Controlled,Total_Item_Requests,EBSCO,0.0,38.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(Out) on the Road: The Radical Joy of Queer Tr...,Ig Publishing,EBSCOhost,Controlled,Unique_Item_Investigations,EBSCO,0.0,38.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,(Out) on the Road: The Radical Joy of Queer Tr...,Ig Publishing,EBSCOhost,Controlled,Unique_Item_Requests,EBSCO,0.0,38.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,(Out) on the Road: The Radical Joy of Queer Tr...,Ig Publishing,EBSCOhost,Controlled,Unique_Title_Investigations,EBSCO,0.0,2.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53437,eMagRes,Wiley,Wiley Online Library,Controlled,Total_Item_Requests,Wiley,7.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53438,eMagRes,Wiley,Wiley Online Library,Controlled,Unique_Item_Investigations,Wiley,3.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53439,eMagRes,Wiley,Wiley Online Library,Controlled,Unique_Item_Requests,Wiley,3.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53440,eMagRes,Wiley,Wiley Online Library,Controlled,Unique_Title_Investigations,Wiley,3.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
newUsage2.shape #Gets the number of rows and columns, for reference

(53442, 17)

In [23]:
newUsage2.columns

Index(['Title', 'Publisher', 'Platform', 'Access_Type', 'Metric_Type',
       'Vendor', 'Jul-2026', 'Aug-2026', 'Reporting_Period_Total',
       'Publisher_ID', 'DOI', 'Proprietary_ID', 'ISBN', 'Print_ISSN',
       'Online_ISSN', 'URI', 'YOP'],
      dtype='object')

In [50]:
#newUsageD.to_sql('TR_C5_Raw_750_End', conn, if_exists = 'replace')

## Limit the Usage to Just the Months Being Loaded

If only a specific range of months are being loaded, comment out the other rows of months by placing the pound sign (#) in front of them. 

You can also use this section to include or remove certain identifier information, such as DOI, ISSN, Proprietary_ID, or other information.

Make sure the fileType matches what you put in the cell above before commenting out rows.

In [24]:
## DR
if fileType == 'DR':
    newUsage2 = newUsage2[['Database', 'Platform', 'Vendor', 'Metric_Type', 'Reporting_Period_Total',
                           'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026',
                           'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                           'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025',
                           'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024',
                           'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024',
                           'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023',
                           'Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023',
                           'Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022',
                           'Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022',
                           'Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021',
                           'Jan-2021', 'Feb-2021', 'Mar-2021', 'Apr-2021', 'May-2021', 'Jun-2021'
                          ]]


## DR_D1
if fileType == 'DR_D1':
    newUsage2 = newUsage2[['Database', 'Publisher', 'Platform', 'Proprietary_ID', 'Vendor', 'Metric_Type', 'Reporting_Period_Total',
                           'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026',
                           'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                           'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025',
                           'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024',
                           'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024',
                           'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023',
                           'Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023',
                           'Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022',
                           'Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022',
                           'Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021',
                          ]]

## PR
if fileType == 'PR':
    newUsage2 = newUsage2[['Platform', 'Vendor', 'Metric_Type', 'Reporting_Period_Total',
                           'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026',
                           'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                           'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025',
                           'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024',
                           'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024',
                           'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023',
                           'Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023',
                           'Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022',
                           'Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022',
                           'Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021',
                           'Jan-2021', 'Feb-2021', 'Mar-2021', 'Apr-2021', 'May-2021', 'Jun-2021',
                          ]]



## PR_P1
if fileType == 'PR_P1':
    newUsage2 = newUsage2[['Platform', 'Vendor', 'Metric_Type', 'Reporting_Period_Total',
                           'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026',
                           #'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                           #'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025',
                           #'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024',
                           #'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024',
                           #'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023',
                           #'Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023',
                           #'Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022',
                           #'Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022',
                           #'Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021',
                          ]]

## TR
if fileType == 'TR':
    newUsage2 = newUsage2[['Title', 'Publisher', 'Platform', 'Print_ISSN', 'Online_ISSN', 'ISBN', 'Vendor', 'Metric_Type',
                           'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026',
                           #'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                           #'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025',
                           #'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024',
                           #'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024',
                           #'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023',
                           #'Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023',
                           #'Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022',
                           #'Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022',
                           #'Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021',
                           #'Jan-2021', 'Feb-2021', 'Mar-2021', 'Apr-2021', 'May-2021', 'Jun-2021'
                          ]]


## TR_J3
if fileType == 'TR_J3':
    newUsage2 = newUsage2[['Title', 'Publisher', 'Platform', 'Print_ISSN', 'Online_ISSN', 'Vendor', 'Access_Type', 'Metric_Type', 'Reporting_Period_Total',
                           'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026',
                           #'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                           #'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025',
                           #'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024',
                           #'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024',
                           #'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023',
                           #'Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023',
                           #'Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022',
                           #'Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022',
                           #'Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021'
                          ]]


## TR_B3
if fileType == 'TR_B3':
    newUsage2 = newUsage2[['Title', 'Publisher', 'Platform', 'ISBN', 'Vendor', 'Access_Type', 'Metric_Type', 'Reporting_Period_Total',
                           'Jul-2026', 'Aug-2026',
                           #'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026',
                           #'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                           #'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025',
                           #'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024',
                           #'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024',
                           #'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023',
                           #'Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023',
                           #'Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022',
                           #'Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022',
                           #'Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021',
                          ]]


## IR_M1
if fileType == 'IR_M1':
    newUsage2 = newUsage2[['Item', 'Publisher', 'Platform', 'DOI', 'Proprietary_ID', 'Vendor', 'Metric_Type', 'Reporting_Period_Total',
                           'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026'
                           #'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                           #'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025',
                           #'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024',
                           #'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024',
                           #'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023',
                           #'Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023',
                           #'Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022',
                           #'Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022',
                           #'Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021',
                          ]]

### Convert Months of Usage to Numbers

Comment out any rows that are not part of newUsage2 before running.

In [25]:
newUsage2['Jul-2026'] = pd.to_numeric(newUsage2['Jul-2026'], errors='coerce')
newUsage2['Aug-2026'] = pd.to_numeric(newUsage2['Aug-2026'], errors='coerce')

## After running this code, all columns that are months should be a number and all other columns should be type "object".

### Edit the TR_J3 and TR_B3 files to account for COUNTER 5.1 changes in terms

COUNTER 5.1 uses "Free to Read" and "Open" for its open access, while COUNTER 5.0 uses "OA_Gold" only. The re-naming conventions below are to improve understanding of what these OA terms actually mean for the content availability.

In [26]:
if fileType == 'TR_J3' or fileType == 'TR_B3':
    newUsage2.loc[newUsage2['Access_Type'] == 'OA_Gold', 'Access_Type'] = 'Publicly_Accessible'
    newUsage2.loc[newUsage2['Access_Type'] == 'Free_To_Read', 'Access_Type'] = 'Temporarily_Open_Only'
    newUsage2.loc[newUsage2['Access_Type'] == 'Open', 'Access_Type'] = 'Publicly_Accessible'
    newUsage2.loc[newUsage2['Access_Type'] == 'Controlled', 'Access_Type'] = 'Subscription_Dependent'

## Fill in Missing Titles, Items, Platforms, Databases, etc.

Missing titles will not appear in the final database.

In [27]:
## IR_M1
if fileType == 'IR_M1':
    newUsage2['Item'] = newUsage2['Item'].fillna('Unknown')

## TR_J3
if fileType == 'TR_J3':
    newUsage2['Title'] = newUsage2['Title'].fillna('Unknown')
    newUsage2['Access_Type'] = newUsage2['Access_Type'].fillna('Unknown')
    newUsage2['Online_ISSN'] = newUsage2['Online_ISSN'].fillna('Unknown')
    newUsage2['Print_ISSN'] = newUsage2['Print_ISSN'].fillna('Unknown')

## TR_B3
if fileType == 'TR_B3':
    newUsage2['Title'] = newUsage2['Title'].fillna('Unknown')
    newUsage2['Access_Type'] = newUsage2['Access_Type'].fillna('Unknown')
    newUsage2['ISBN'] = newUsage2['ISBN'].fillna('Unknown')

## DR_D1
if fileType == 'DR_D1':
    newUsage2['Database'] = newUsage2['Database'].fillna('Unknown')

## PR_P1
if fileType == 'PR_P1':
    newUsage2['Platform'] = newUsage2['Platform'].fillna('Unknown')

## TR
if fileType == 'TR':
    newUsage2['Title'] = newUsage2['Title'].fillna('Unknown')
    newUsage2['Online_ISSN'] = newUsage2['Online_ISSN'].fillna('Unknown')
    newUsage2['Print_ISSN'] = newUsage2['Print_ISSN'].fillna('Unknown')
    newUsage2['ISBN'] = newUsage2['ISBN'].fillna('Unknown')

## DR
if fileType == 'DR':
    newUsage2['Database'] = newUsage2['Database'].fillna('Unknown')

## PR
if fileType == 'PR':
    newUsage2['Platform'] = newUsage2['Platform'].fillna('Unknown')

### Check for Missing Information

This can include off-centered headers that lead to inaccurate information, months not included in the data files, and other irregularities. Vendors whose usage appears in this cell will need their usage files checked for formatting.

Common reasons a vendor may appear below:

1. Usage started midway through a date period. As long as this is confirmed, this is okay and the code will still pick up the partial period's usage.
2. The vendor does not provide information on Platform for at least one of the items in the report. This can happen, but needs to be confirmed true, as this is rare. Once confirmed, the code will still pick up the usage.
3. The file was loaded incorrectly. Usually, this means the COUNTER type was incorrect (such as loading a COUNTER 5 report in the COUNTER 5.1 format). Either relocate the file or adjust the header to work properly, then re-load the data (or at least that file(s) if working with lots of info) and run again.

In [28]:
## Off-Centered Check

offCenter = newUsage2.loc[newUsage2['Platform'].isnull()]

offCenter['Vendor'].unique()


array([], dtype=object)

In [29]:
#newUsage

In [30]:
## Missing Months Checks - Only one row at a time can be run

#missingMonths = newUsage2.loc[newUsage2['Jan-2025'].isnull() | newUsage2['Feb-2025'].isnull() | newUsage2['Mar-2025'].isnull() | newUsage2['Apr-2025'].isnull() | newUsage2['May-2025'].isnull() | newUsage2['Jun-2025'].isnull()]
#missingMonths = newUsage2.loc[newUsage2['Jul-2024'].isnull() | newUsage2['Aug-2024'].isnull() | newUsage2['Sep-2024'].isnull() | newUsage2['Oct-2024'].isnull() | newUsage2['Nov-2024'].isnull() | newUsage2['Dec-2024'].isnull()]
#missingMonths = newUsage2.loc[newUsage2['Jan-2024'].isnull() | newUsage2['Feb-2024'].isnull() | newUsage2['Mar-2024'].isnull() | newUsage2['Apr-2024'].isnull() | newUsage2['May-2024'].isnull() | newUsage2['Jun-2024'].isnull()]
#missingMonths = newUsage2.loc[newUsage2['Jul-2023'].isnull() | newUsage2['Aug-2023'].isnull() | newUsage2['Sep-2023'].isnull() | newUsage2['Oct-2023'].isnull() | newUsage2['Nov-2023'].isnull() | newUsage2['Dec-2023'].isnull()]
#missingMonths = newUsage2.loc[newUsage2['Jan-2023'].isnull() | newUsage2['Feb-2023'].isnull() | newUsage2['Mar-2023'].isnull() | newUsage2['Apr-2023'].isnull() | newUsage2['May-2023'].isnull() | newUsage2['Jun-2023'].isnull()]
#missingMonths = newUsage2.loc[newUsage2['Jul-2022'].isnull() | newUsage2['Aug-2022'].isnull() | newUsage2['Sep-2022'].isnull() | newUsage2['Oct-2022'].isnull() | newUsage2['Nov-2022'].isnull() | newUsage2['Dec-2022'].isnull()]
#missingMonths = newUsage2.loc[newUsage2['Jan-2022'].isnull() | newUsage2['Feb-2022'].isnull() | newUsage2['Mar-2022'].isnull() | newUsage2['Apr-2022'].isnull() | newUsage2['May-2022'].isnull() | newUsage2['Jun-2022'].isnull()]
#missingMonths = newUsage2.loc[newUsage2['Jul-2021'].isnull() | newUsage2['Aug-2021'].isnull() | newUsage2['Sep-2021'].isnull() | newUsage2['Oct-2021'].isnull() | newUsage2['Nov-2021'].isnull() | newUsage2['Dec-2021'].isnull()]
#missingMonths = newUsage2.loc[newUsage2['Jan-2021'].isnull() | newUsage2['Feb-2021'].isnull() | newUsage2['Mar-2021'].isnull() | newUsage2['Apr-2021'].isnull() | newUsage2['May-2021'].isnull() | newUsage2['Jun-2021'].isnull()]

#missingMonths['Vendor'].unique()

### Filter the Usage Down to Relevant Metrics

#### As of March 2026, the following metrics are used for various usage-related jobs and should be in the "isin" block below: Unique_Item_Requests, Total_Item_Requests, Unique_Title_Requests, Searches_Platform, Searches_Regular, Searches_Automated, Searches_Federated, No_License, Limit_Exceeded. 

If projects require new metrics or if metrics are not being used, you can edit the "isin" block below. 

Don't worry if your reports are missing some of these; the code will still work as intended.

In [31]:
newUsage3 = newUsage2[newUsage2['Metric_Type'].isin(['Unique_Item_Requests', 'Total_Item_Requests', 'Unique_Title_Requests', 'Searches_Platform', 'Searches_Regular', 'Searches_Automated', 'Searches_Federated', 'No_License', 'Limit_Exceeded'])]

In [32]:
newUsage3.shape

(20319, 10)

# Upload Data to the Database

## Load Existing Data from the Database

The "oldUsage" line below reads in the data from the Raw file stored in the database already. The oldAndNew lines combine the new and old usage together.

#### Make sure the file type is correct before continuing; otherwise, you'll be pulling from the wrong place.

In [33]:
if fileType == 'PR' or fileType == 'TR' or fileType == 'DR':
    queryToRead = r'SELECT * FROM '+fileType+'_Archive;'
else:
    queryToRead = r'SELECT * FROM '+fileType+'_Raw;'

queryToRead

oldUsage = pd.read_sql_query(queryToRead, conn)

### Existing Data Repair Shop

If there is any data in oldUsage that needs to be edited, use the block below to make those edits. Otherwise, comment out and/or skip the next block, picking up with the oldAndNew cell.

In [34]:
#oldUsage = oldUsage.reset_index()

#for i in range(len(oldUsage['Title'])):
#    if oldUsage.loc[i, 'Vendor'] == 'T&Febooks':
#        oldUsage.loc[i, 'Jan-2026'] = 0
#        oldUsage.loc[i, 'Feb-2026'] = 0
#        oldUsage.loc[i, 'Mar-2026'] = 0
#        oldUsage.loc[i, 'Apr-2026'] = 0
#        oldUsage.loc[i, 'May-2026'] = 0
#        oldUsage.loc[i, 'Jun-2026'] = 0

In [35]:
oldAndNew = pd.concat([oldUsage, newUsage3], ignore_index = True)

In [36]:
oldAndNew.columns

Index(['index', 'Title', 'Access_Type', 'ISBN', 'Vendor', 'Metric_Type',
       'Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021',
       'Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022',
       'Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022',
       'Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023',
       'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023',
       'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024',
       'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024',
       'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025',
       'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
       'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026',
       'Reporting_Period_Total', 'Publisher', 'Platform', 'Jul-2026',
       'Aug-2026'],
      dtype='object')

### All Data Repair Shop

Use this block for data edits that need to span both the old and new usage, such as naming conventions or general format consistencies. If none are needed, comment out and/or skip the next block, picking up with the Metric Type cell.

In [37]:
oldAndNew['Vendor'] = oldAndNew['Vendor'].replace({'Onepetro':'OnePetro', 'ProjectMUSE':'ProjectMuse'})

#oldAndNew['Online_ISSN'] = oldAndNew['Online_ISSN'].str.replace("\W", '', regex=True)
#oldAndNew['Print_ISSN'] = oldAndNew['Print_ISSN'].str.replace("\W", '', regex=True)
oldAndNew['ISBN'] = oldAndNew['ISBN'].str.replace("\W", '', regex=True)

In [38]:
#oldAndNew.to_sql('oldAndNew TR to Replace TR_Archive', conn, if_exists = 'replace')
#oldAndNew = pd.read_sql_query(r'SELECT * FROM `oldAndNew TR to Replace TR_Archive`;', conn)

In [39]:
oldAndNew = oldAndNew[oldAndNew['Metric_Type'].isin(['Unique_Item_Requests', 'Total_Item_Requests', 'Unique_Title_Requests', 'Searches_Platform', 'Searches_Regular', 'Searches_Automated', 'Searches_Federated', 'No_License', 'Limit_Exceeded'])] #This can be removed if the metric limits are already accounted for above.

oldAndNew = oldAndNew.drop_duplicates()

oldAndNew.shape #Note: this is not the number of rows that will appear in the database. This is how many rows will be combined together in the next cell.

(604589, 71)

### Check Vendor Usage Calculations

Making sure a vendor has all of its usage? Use the cells below to check this. Otherwise, you can skip to the next section.

In [42]:
test = oldAndNew[oldAndNew['Vendor'] == 'T&Febooks']
test2 = test[test['Metric_Type'] == 'Unique_Title_Requests']

#test2

In [43]:
summaryTest = test2.groupby(['Vendor'], dropna = False).agg( #This is where the first row options from the cell above should go
    {#'Jan-2021':lambda x: x.sum(min_count = 1),
     #'Feb-2021':lambda x: x.sum(min_count = 1),
     #'Mar-2021':lambda x: x.sum(min_count = 1),
     #'Apr-2021':lambda x: x.sum(min_count = 1),
     #'May-2021':lambda x: x.sum(min_count = 1),
     #'Jun-2021':lambda x: x.sum(min_count = 1),
     'Jul-2021':lambda x: x.sum(min_count = 1),
     'Aug-2021':lambda x: x.sum(min_count = 1),
     'Sep-2021':lambda x: x.sum(min_count = 1),
     'Oct-2021':lambda x: x.sum(min_count = 1),
     'Nov-2021':lambda x: x.sum(min_count = 1),
     'Dec-2021':lambda x: x.sum(min_count = 1),
     'Jan-2022':lambda x: x.sum(min_count = 1),
     'Feb-2022':lambda x: x.sum(min_count = 1),
     'Mar-2022':lambda x: x.sum(min_count = 1),
     'Apr-2022':lambda x: x.sum(min_count = 1),
     'May-2022':lambda x: x.sum(min_count = 1),
     'Jun-2022':lambda x: x.sum(min_count = 1),
     'Jul-2022':lambda x: x.sum(min_count = 1),
     'Aug-2022':lambda x: x.sum(min_count = 1),
     'Sep-2022':lambda x: x.sum(min_count = 1),
     'Oct-2022':lambda x: x.sum(min_count = 1),
     'Nov-2022':lambda x: x.sum(min_count = 1),
     'Dec-2022':lambda x: x.sum(min_count = 1),
     'Jan-2023':lambda x: x.sum(min_count = 1),
     'Feb-2023':lambda x: x.sum(min_count = 1),
     'Mar-2023':lambda x: x.sum(min_count = 1),
     'Apr-2023':lambda x: x.sum(min_count = 1),
     'May-2023':lambda x: x.sum(min_count = 1),
     'Jun-2023':lambda x: x.sum(min_count = 1),
     'Jul-2023':lambda x: x.sum(min_count = 1),
     'Aug-2023':lambda x: x.sum(min_count = 1),
     'Sep-2023':lambda x: x.sum(min_count = 1),
     'Oct-2023':lambda x: x.sum(min_count = 1),
     'Nov-2023':lambda x: x.sum(min_count = 1),
     'Dec-2023':lambda x: x.sum(min_count = 1),
     'Jan-2024':lambda x: x.sum(min_count = 1),
     'Feb-2024':lambda x: x.sum(min_count = 1),
     'Mar-2024':lambda x: x.sum(min_count = 1),
     'Apr-2024':lambda x: x.sum(min_count = 1),
     'May-2024':lambda x: x.sum(min_count = 1),
     'Jun-2024':lambda x: x.sum(min_count = 1),
     'Jul-2024':lambda x: x.sum(min_count = 1),
     'Aug-2024':lambda x: x.sum(min_count = 1),
     'Sep-2024':lambda x: x.sum(min_count = 1),
     'Oct-2024':lambda x: x.sum(min_count = 1),
     'Nov-2024':lambda x: x.sum(min_count = 1),
     'Dec-2024':lambda x: x.sum(min_count = 1),
     'Jan-2025':lambda x: x.sum(min_count = 1),
     'Feb-2025':lambda x: x.sum(min_count = 1),
     'Mar-2025':lambda x: x.sum(min_count = 1),
     'Apr-2025':lambda x: x.sum(min_count = 1),
     'May-2025':lambda x: x.sum(min_count = 1),
     'Jun-2025':lambda x: x.sum(min_count = 1),
     'Jul-2025':lambda x: x.sum(min_count = 1),
     'Aug-2025':lambda x: x.sum(min_count = 1),
     'Sep-2025':lambda x: x.sum(min_count = 1),
     'Oct-2025':lambda x: x.sum(min_count = 1),
     'Nov-2025':lambda x: x.sum(min_count = 1),
     'Dec-2025':lambda x: x.sum(min_count = 1),
     'Jan-2026':lambda x: x.sum(min_count = 1),
     'Feb-2026':lambda x: x.sum(min_count = 1),
     'Mar-2026':lambda x: x.sum(min_count = 1),
     'Apr-2026':lambda x: x.sum(min_count = 1),
     'May-2026':lambda x: x.sum(min_count = 1),
     'Jun-2026':lambda x: x.sum(min_count = 1),
     'Jul-2026':lambda x: x.sum(min_count = 1),
     'Aug-2026':lambda x: x.sum(min_count = 1)
    }, min_count = 1).reset_index()

In [44]:
pd.set_option('display.max_columns', None)
summaryTest

,Vendor,Jul-2021,Aug-2021,Sep-2021,Oct-2021,Nov-2021,Dec-2021,Jan-2022,Feb-2022,Mar-2022,Apr-2022,May-2022,Jun-2022,Jul-2022,Aug-2022,Sep-2022,Oct-2022,Nov-2022,Dec-2022,Jan-2023,Feb-2023,Mar-2023,Apr-2023,May-2023,Jun-2023,Jul-2023,Aug-2023,Sep-2023,Oct-2023,Nov-2023,Dec-2023,Jan-2024,Feb-2024,Mar-2024,Apr-2024,May-2024,Jun-2024,Jul-2024,Aug-2024,Sep-2024,Oct-2024,Nov-2024,Dec-2024,Jan-2025,Feb-2025,Mar-2025,Apr-2025,May-2025,Jun-2025,Jul-2025,Aug-2025,Sep-2025,Oct-2025,Nov-2025,Dec-2025,Jan-2026,Feb-2026,Mar-2026,Apr-2026,May-2026,Jun-2026,Jul-2026,Aug-2026
0,T&Febooks,145.0,274.0,1499.0,2949.0,2800.0,1104.0,1265.0,2334.0,2136.0,2352.0,2180.0,829.0,674.0,957.0,2351.0,2600.0,3329.0,1656.0,1445.0,3727.0,2796.0,2796.0,2101.0,1035.0,954.0,1592.0,2657.0,2718.0,2915.0,1626.0,2254.0,4168.0,2922.0,3749.0,2148.0,935.0,1367.0,2727.0,5378.0,4199.0,3279.0,1828.0,2395.0,3763.0,2690.0,3039.0,2015.0,1075.0,1021.0,1922.0,3691.0,3103.0,2560.0,1495.0,2631.0,3876.0,2879.0,3003.0,1738.0,1151.0,972.0,2506.0


# Grab the Necessary First Line based on the Report Type

The summaryTable code below will require manually changing the first row. The next cell contains all of the options for what that first row could be based on the file type.

Copy everything inside of the quotes (the quotes are not needed) and paste it in the first row of the summaryTable function three cells below.

In [45]:
### Options:


### DR
r"oldAndNew.groupby(['Database', 'Vendor', 'Metric_Type']).agg("


### DR_D1
r"oldAndNew.groupby(['Database', 'Vendor', 'Metric_Type']).agg("


### PR
r"oldAndNew.groupby(['Platform', 'Vendor', 'Metric_Type']).agg("


### PR_P1
r"oldAndNew.groupby(['Platform', 'Vendor', 'Metric_Type']).agg("


### TR
r"oldAndNew.groupby(['Title', 'Online_ISSN', 'Print_ISSN', 'ISBN', 'Vendor', 'Metric_Type'], dropna = False).agg("


### TR_J1
r"oldAndNew.groupby(['Title', 'Vendor', 'Metric_Type']).agg("


### TR_J3
r"oldAndNew.groupby(['Title', 'Access_Type', 'Online_ISSN', 'Print_ISSN', 'Vendor', 'Metric_Type']).agg("


### TR_J4
r"oldAndNew.groupby(['Title', 'YOP', 'Vendor', 'Metric_Type']).agg("


### TR_B3
r"oldAndNew.groupby(['Title', 'Access_Type', 'ISBN', 'Vendor', 'Metric_Type']).agg("


### IR_M1
r"oldAndNew.groupby(['Item', 'Vendor', 'Metric_Type']).agg("

"oldAndNew.groupby(['Item', 'Vendor', 'Metric_Type']).agg("

### Combine Data

This may take significant time, especially for very large reports like Title and Item reports. 

#### Remember: Comment out any rows in the cell below that are not part of the overall database!

In [46]:
## Aggregate All Usage
summaryTable = oldAndNew.groupby(['Title', 'Access_Type', 'ISBN', 'Vendor', 'Metric_Type']).agg( #This is where the first row options from the cell above should go. Don't overwrite the summaryTable = portion!
    {#'Jan-2021':lambda x: x.sum(min_count = 1),
     #'Feb-2021':lambda x: x.sum(min_count = 1),
     #'Mar-2021':lambda x: x.sum(min_count = 1),
     #'Apr-2021':lambda x: x.sum(min_count = 1),
     #'May-2021':lambda x: x.sum(min_count = 1),
     #'Jun-2021':lambda x: x.sum(min_count = 1),
     'Jul-2021':lambda x: x.sum(min_count = 1),
     'Aug-2021':lambda x: x.sum(min_count = 1),
     'Sep-2021':lambda x: x.sum(min_count = 1),
     'Oct-2021':lambda x: x.sum(min_count = 1),
     'Nov-2021':lambda x: x.sum(min_count = 1),
     'Dec-2021':lambda x: x.sum(min_count = 1),
     'Jan-2022':lambda x: x.sum(min_count = 1),
     'Feb-2022':lambda x: x.sum(min_count = 1),
     'Mar-2022':lambda x: x.sum(min_count = 1),
     'Apr-2022':lambda x: x.sum(min_count = 1),
     'May-2022':lambda x: x.sum(min_count = 1),
     'Jun-2022':lambda x: x.sum(min_count = 1),
     'Jul-2022':lambda x: x.sum(min_count = 1),
     'Aug-2022':lambda x: x.sum(min_count = 1),
     'Sep-2022':lambda x: x.sum(min_count = 1),
     'Oct-2022':lambda x: x.sum(min_count = 1),
     'Nov-2022':lambda x: x.sum(min_count = 1),
     'Dec-2022':lambda x: x.sum(min_count = 1),
     'Jan-2023':lambda x: x.sum(min_count = 1),
     'Feb-2023':lambda x: x.sum(min_count = 1),
     'Mar-2023':lambda x: x.sum(min_count = 1),
     'Apr-2023':lambda x: x.sum(min_count = 1),
     'May-2023':lambda x: x.sum(min_count = 1),
     'Jun-2023':lambda x: x.sum(min_count = 1),
     'Jul-2023':lambda x: x.sum(min_count = 1),
     'Aug-2023':lambda x: x.sum(min_count = 1),
     'Sep-2023':lambda x: x.sum(min_count = 1),
     'Oct-2023':lambda x: x.sum(min_count = 1),
     'Nov-2023':lambda x: x.sum(min_count = 1),
     'Dec-2023':lambda x: x.sum(min_count = 1),
     'Jan-2024':lambda x: x.sum(min_count = 1),
     'Feb-2024':lambda x: x.sum(min_count = 1),
     'Mar-2024':lambda x: x.sum(min_count = 1),
     'Apr-2024':lambda x: x.sum(min_count = 1),
     'May-2024':lambda x: x.sum(min_count = 1),
     'Jun-2024':lambda x: x.sum(min_count = 1),
     'Jul-2024':lambda x: x.sum(min_count = 1),
     'Aug-2024':lambda x: x.sum(min_count = 1),
     'Sep-2024':lambda x: x.sum(min_count = 1),
     'Oct-2024':lambda x: x.sum(min_count = 1),
     'Nov-2024':lambda x: x.sum(min_count = 1),
     'Dec-2024':lambda x: x.sum(min_count = 1),
     'Jan-2025':lambda x: x.sum(min_count = 1),
     'Feb-2025':lambda x: x.sum(min_count = 1),
     'Mar-2025':lambda x: x.sum(min_count = 1),
     'Apr-2025':lambda x: x.sum(min_count = 1),
     'May-2025':lambda x: x.sum(min_count = 1),
     'Jun-2025':lambda x: x.sum(min_count = 1),
     'Jul-2025':lambda x: x.sum(min_count = 1),
     'Aug-2025':lambda x: x.sum(min_count = 1),
     'Sep-2025':lambda x: x.sum(min_count = 1),
     'Oct-2025':lambda x: x.sum(min_count = 1),
     'Nov-2025':lambda x: x.sum(min_count = 1),
     'Dec-2025':lambda x: x.sum(min_count = 1),
     'Jan-2026':lambda x: x.sum(min_count = 1),
     'Feb-2026':lambda x: x.sum(min_count = 1),
     'Mar-2026':lambda x: x.sum(min_count = 1),
     'Apr-2026':lambda x: x.sum(min_count = 1),
     'May-2026':lambda x: x.sum(min_count = 1),
     'Jun-2026':lambda x: x.sum(min_count = 1),
     'Jul-2026':lambda x: x.sum(min_count = 1),
     'Aug-2026':lambda x: x.sum(min_count = 1),
    }, min_count = 1).reset_index()

### Add in Reporting_Period_Total

In [47]:
summaryTable['Reporting_Period_Total'] = summaryTable[['Aug-2026', 'Jul-2026', 'Jun-2026', 'May-2026', 'Apr-2026', 'Mar-2026', 'Feb-2026', 'Jan-2026', 'Dec-2025', 'Nov-2025', 'Oct-2025', 'Sep-2025', 'Aug-2025', 'Jul-2025', 'Jun-2025', 'May-2025', 'Apr-2025', 'Mar-2025', 'Feb-2025', 'Jan-2025', 'Dec-2024', 'Nov-2024', 'Oct-2024', 'Sep-2024', 'Aug-2024', 'Jul-2024', 'Jun-2024', 'May-2024', 'Apr-2024', 'Mar-2024', 'Feb-2024', 'Jan-2024', 'Dec-2023', 'Nov-2023', 'Oct-2023', 'Sep-2023', 'Aug-2023', 'Jul-2023', 'Jun-2023', 'May-2023', 'Apr-2023', 'Mar-2023', 'Feb-2023', 'Jan-2023', 'Dec-2022', 'Nov-2022', 'Oct-2022', 'Sep-2022', 'Aug-2022', 'Jul-2022', 'Jun-2022', 'May-2022', 'Apr-2022', 'Mar-2022', 'Feb-2022', 'Jan-2022', 'Dec-2021', 'Nov-2021', 'Oct-2021', 'Sep-2021', 'Aug-2021', 'Jul-2021']].sum(axis = 1)



## Remember: Do not include any columns that are not part of the overall database! The oldest columns are in the back.

# Place unnecessary columns next to the below pound signs:

#, 'Jun-2021', 'May-2021', 'Apr-2021', 'Mar-2021', 'Feb-2021', 'Jan-2021'
#

summaryTable.shape #This is the number of rows going to the database.

(596135, 68)

## Update the Raw Database

### WARNING: Only do this step once all other cells have been run! Once a database has been overwritten, it can not be undone.

In [48]:
#if fileType == 'PR' or fileType == 'TR' or fileType == 'DR':
#    tableName = r''+fileType+'_Archive'
#else:
#    tableName = r''+fileType+'_Raw'

tableName = r''+fileType+'_VolBooks'

tableName

'TR_B3_VolBooks'

#### Final Data Cleaning Step 

This cell takes care of two edge cases that need to be filtered:

1. Nature data is removed entirely. This is because SpringerAll and Nature are the same.
2. Any titles with 0 usage over the entire period are removed. This may occur if data is input incorrectly, then cleaned up later (for example, file formatting causes blank rows or header rows to be treated as titles, but are caught later). If the thing has no usage, it does not need to take up space in the database.

Both of these are safety last checks, so if you know neither are part of your data, you can skip the next cell.

In [49]:
summaryTable = summaryTable[summaryTable['Vendor'] != 'Nature']

summaryTable = summaryTable[summaryTable['Reporting_Period_Total'] != 0]

summaryTable.shape

(596135, 68)

In [50]:
summaryTable.to_sql(tableName, conn, if_exists = 'replace')

596135

## Update the Long Database (One Row Per Month)

This takes the raw data and converts it into a longer format, where each month has its own row instead of its own column. This is used for the line graphs in the PowerBI dashboard.

## WARNING: Only do this step once all other cells have been run! Once a database has been overwritten, it can not be undone.

In [51]:
tableNameLong = r''+fileType+'_Long'

tableNameLong

'TR_B3_Long'

### Pivot and Prepare the Data

In [52]:
longTable = summaryTable.pivot_longer(
    column_names = '[A-z][A-z][A-z]-[0-9][0-9][0-9][0-9]',
    names_to = 'Month',
    values_to = 'Usage')

longTable = longTable[longTable['Usage'].notna()]

longTable['Month'] = pd.to_datetime(longTable['Month'])

In [12]:
longTable['FY'] = longTable['Month'].map(lambda x: x.year if x.month < 7 else x.year+1).astype(int).astype(int)

#### Select Columns

In [13]:
if fileType == 'DR':
    longTable = longTable[['Database', 'Vendor', 'Metric_Type', 'Reporting_Period_Total', 'Month', 'Usage', 'FY']]
elif fileType == 'DR_D1':
    longTable = longTable[['Database', 'Vendor', 'Metric_Type', 'Reporting_Period_Total', 'Month', 'Usage', 'FY']]
elif fileType == 'PR':
    longTable = longTable[['Platform', 'Vendor', 'Metric_Type', 'Reporting_Period_Total', 'Month', 'Usage', 'FY']]
elif fileType == 'PR_P1':
    longTable = longTable[['Platform', 'Vendor', 'Metric_Type', 'Reporting_Period_Total', 'Month', 'Usage', 'FY']]
elif fileType == 'TR':
    longTable = longTable[['Title', 'Online_ISSN', 'Print_ISSN', 'ISBN', 'Vendor', 'Metric_Type', 'Reporting_Period_Total', 'Month', 'Usage', 'FY']]
elif fileType == 'TR_J3':
    longTable = longTable[['Title', 'Access_Type', 'Online_ISSN', 'Print_ISSN', 'Vendor', 'Metric_Type', 'Reporting_Period_Total', 'Month', 'Usage', 'FY']]
elif fileType == 'TR_B3':
    longTable = longTable[['Title', 'Access_Type', 'ISBN', 'Vendor', 'Metric_Type', 'Reporting_Period_Total', 'Month', 'Usage', 'FY']]
elif fileType == 'IR_M1':
    longTable = longTable[['Item', 'Vendor', 'Metric_Type', 'Reporting_Period_Total', 'Month', 'Usage', 'FY']]

In [14]:
longTable.shape

(20896044, 9)

### Select Only the Last 3 Years for PowerBI Purposes

This needs to include all months in the usage period being gathered. For example, if this is being run in May 2026, then all usage since January 2023 needs to be accounted for, since May 2023 is in the Jan-Jun 2023 usage window.

In [22]:
if fileType == 'PR' or fileType == 'TR' or fileType == 'DR':
    longTableSmall = longTable[longTable['Month'] >= '2023-07-01']
else:
    longTableSmall = longTable
    
longTableSmall.shape

(20896044, 9)

In [15]:
longTableSmall.to_sql(tableNameLong, conn, if_exists = 'replace')

20896044

## Clean the Standard Report Long Tables for PowerBI Integration

The TR_B3, TR_J3, PR_P1, DR_D1, and IR_M1 reports (or the "standard views" as COUNTER refers to them) are currently specifically used for annual statistics reporting purposes. To save space in the PowerBI report, the large Long Tables will be shortened to only include relevant information for annual statistics reporting.

In [48]:
if fileType == 'TR_B3' or fileType == 'TR_J3' or fileType == 'DR_D1' or fileType == 'PR_P1' or fileType == 'IR_M1':
    powerBILongTable = longTable.groupby(['Vendor', 'Metric_Type', 'Month'])['Usage'].sum().reset_index()

powerBILongTable['Month'] = pd.to_datetime(powerBILongTable['Month'])

powerBILongTable['FY'] = powerBILongTable['Month'].map(lambda x: x.year if x.month < 7 else x.year+1).astype(int).astype(int)

powerBILongTable

,Vendor,Metric_Type,Month,Usage,FY
0,AAAS,Searches_Platform,2021-07-01,0.0,2022
1,AAAS,Searches_Platform,2021-08-01,5.0,2022
2,AAAS,Searches_Platform,2021-09-01,155.0,2022
3,AAAS,Searches_Platform,2021-10-01,124.0,2022
4,AAAS,Searches_Platform,2021-11-01,146.0,2022
...,...,...,...,...,...
23375,WorldScientific,Unique_Title_Requests,2026-02-01,1.0,2026
23376,WorldScientific,Unique_Title_Requests,2026-03-01,3.0,2026
23377,WorldScientific,Unique_Title_Requests,2026-04-01,4.0,2026
23378,WorldScientific,Unique_Title_Requests,2026-05-01,2.0,2026


In [49]:
powerBILongTableName = r''+fileType+'_Long_PowerBI'

powerBILongTableName

'PR_P1_Long_PowerBI'

In [50]:
powerBILongTable.to_sql(powerBILongTableName, conn, if_exists = 'replace')

23380

## Add in Columns for PowerBI Integration

These columns are specifically for PowerBI and other external uses FOR MASTER TABLES ONLY. This applies only to the wide table, not the long table.

### Note: Change the months in the cell below based on when the three-year cutoff exists.

In [ ]:
## Only Get the Last 3 Years of Usage

powerBITable = summaryTable

if fileType == 'TR':
    powerBITable = powerBITable[['Title', 'Online_ISSN', 'Print_ISSN', 'ISBN', 'Vendor', 'Metric_Type',
                                 'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023',
                                 'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024',
                                 'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024',
                                 'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025',
                                 'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                                 'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026',]]
elif fileType == 'PR':
    powerBITable = powerBITable[['Platform', 'Vendor', 'Metric_Type', 
                                 'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023', 
                                 'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024', 
                                 'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024', 
                                 'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025', 
                                 'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                                 'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026']]

elif fileType == 'DR':
    powerBITable = powerBITable[['Database', 'Vendor', 'Metric_Type', 
                                 'Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023', 
                                 'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024', 
                                 'Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024', 
                                 'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025', 
                                 'Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025',
                                 'Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026']]    
else:
    powerBITable = powerBITable

    
if fileType == 'TR' or fileType == 'PR' or fileType == 'DR':
    powerBITable['Reporting_Period_Total'] = powerBITable[['Jun-2026', 'May-2026', 'Apr-2026', 'Mar-2026', 'Feb-2026', 'Jan-2026', 'Dec-2025', 'Nov-2025', 'Oct-2025', 'Sep-2025', 'Aug-2025', 'Jul-2025', 'Jun-2025', 'May-2025', 'Apr-2025', 'Mar-2025', 'Feb-2025', 'Jan-2025', 'Dec-2024', 'Nov-2024', 'Oct-2024', 'Sep-2024', 'Aug-2024', 'Jul-2024', 'Jun-2024', 'May-2024', 'Apr-2024', 'Mar-2024', 'Feb-2024', 'Jan-2024', 'Dec-2023', 'Nov-2023', 'Oct-2023', 'Sep-2023', 'Aug-2023', 'Jul-2023']].sum(axis = 1)

powerBITable.shape

In [ ]:
## Clear out the 0 Usage Items

powerBITable = powerBITable[powerBITable['Reporting_Period_Total'] > 0]

powerBITable.shape

In [ ]:
## Usage Per Year

if fileType == 'TR' or fileType == 'PR' or fileType == 'DR':
    powerBITable['Usage_Per_Year'] = (powerBITable['Reporting_Period_Total']) / 3
else:
    powerBITable['Usage_Per_Year'] = (powerBITable['Reporting_Period_Total']) / 5

### Calculate Annual Usage

#### Note: If a year has fewer than all 12 months, then only include those months in the double brackets. Make sure those brackets end with the .sum(axis = 1) part of the code, as this piece is what adds everything up.

Any years not part of the usage at all should be commented out.

In [ ]:
## Annual Confirmed Usage

if fileType != 'TR' and fileType != 'DR' and fileType != 'PR':
    powerBITable['Usage_2021'] = powerBITable[['Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021']].sum(axis = 1)
    powerBITable['Usage_2022'] = powerBITable[['Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022', 'Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022']].sum(axis = 1)
    powerBITable['Usage_2023'] = powerBITable[['Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023', 'Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023']].sum(axis = 1)

if fileType == 'TR' or fileType == 'DR' or fileType == 'PR':
    powerBITable['Usage_2023'] = powerBITable[['Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023']].sum(axis = 1)


powerBITable['Usage_2024'] = powerBITable[['Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024', 'Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024']].sum(axis = 1)
powerBITable['Usage_2025'] = powerBITable[['Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025', 'Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025']].sum(axis = 1)
powerBITable['Usage_2026'] = powerBITable[['Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026']].sum(axis = 1)



powerBITable

## Update the Cleaned PowerBI Database

WARNING: Only do this step once all other cells have been run! Once a database has been overwritten, it can not be undone.

In [ ]:
powerBITable.to_sql(fileType, conn, if_exists='replace')

## Create Emergency Backup Excel File

If the database is ever overwritten, damaged, or in any way loses its information, the emergency files below can be read in to create a new table and/or refresh the data. Keep this updated as new data is properly added to each database.

To save storage space, only the wide archival database will be saved as a backup. It can be converted to long format and PowerBI filtered format (if needed) using the long database cell above.

Note that this may take several minutes for large Excel files, such as those for Title and Item reports.

### If your file would be larger than 1,048,576 rows, skip this part, as Excel is unable to handle any additional information beyond that number of rows.

In [ ]:
folderPath = r"C:\Users\aharrima\OneDrive - University of Tennessee\Documents\Usage Database Update\Emergency Backups" # Change this path for your computer as needed.

directory = os.chdir(folderPath)


# Export Data
fileExportName = r''+fileType+' COUNTER Usage Raw Backup File.xlsx'

summaryTable.to_excel(fileExportName, index = False)

## Update Usage Information File

These files track how much usage is known, what has yet to be gathered, and what is lost usage. If the cell contains "True", then the usage is in the database. If it is "False", then either there was no usage or it was never gathered.

In [ ]:
## File Tracker Columns - For Export and External Gathering Only

fileTracker = pd.DataFrame()

# Get the Type of Item Being Investigated based on File Type

if fileType == 'DR' or fileType == 'DR_D1':
    fileTracker['Database'] = powerBITable['Database']
    
if fileType == 'PR' or fileType == 'PR_P1':
    fileTracker['Platform'] = powerBITable['Platform']

if fileType == 'TR':
    fileTracker['Title'] = powerBITable['Title']
    fileTracker['Online_ISSN'] = powerBITable['Online_ISSN']
    fileTracker['Print_ISSN'] = powerBITable['Print_ISSN']
    fileTracker['ISBN'] = powerBITable['ISBN']

if fileType == 'TRB3': 
    fileTracker['Title'] = powerBITable['Title']
    fileTracker['ISBN'] = powerBITable['ISBN']

if fileType == 'TRJ3':
    fileTracker['Title'] = powerBITable['Title']
    fileTracker['Online_ISSN'] = powerBITable['Online_ISSN']
    fileTracker['Print_ISSN'] = powerBITable['Print_ISSN']

if fileType == 'IR_M1':
    fileTracker['Item'] = powerBITable['Item']

fileTracker['Vendor'] = powerBITable['Vendor']

### Don't forget to comment out any six-month chunk that is not in the powerBITable file from the cell below.

In [ ]:
if fileType != 'TR' and fileType != 'DR' and fileType != 'PR':
    fileTracker['Jul_Dec_2021'] = powerBITable[['Jul-2021', 'Aug-2021', 'Sep-2021', 'Oct-2021', 'Nov-2021', 'Dec-2021']].notnull().any(axis = 1)
    fileTracker['Jan_Jun_2022'] = powerBITable[['Jan-2022', 'Feb-2022', 'Mar-2022', 'Apr-2022', 'May-2022', 'Jun-2022']].notnull().any(axis = 1)
    fileTracker['Jul_Dec_2022'] = powerBITable[['Jul-2022', 'Aug-2022', 'Sep-2022', 'Oct-2022', 'Nov-2022', 'Dec-2022']].notnull().any(axis = 1)
    fileTracker['Jan_Jun_2023'] = powerBITable[['Jan-2023', 'Feb-2023', 'Mar-2023', 'Apr-2023', 'May-2023', 'Jun-2023']].notnull().any(axis = 1)
    
fileTracker['Jul_Dec_2023'] = powerBITable[['Jul-2023', 'Aug-2023', 'Sep-2023', 'Oct-2023', 'Nov-2023', 'Dec-2023']].notnull().any(axis = 1)
fileTracker['Jan_Jun_2024'] = powerBITable[['Jan-2024', 'Feb-2024', 'Mar-2024', 'Apr-2024', 'May-2024', 'Jun-2024']].notnull().any(axis = 1)
fileTracker['Jul_Dec_2024'] = powerBITable[['Jul-2024', 'Aug-2024', 'Sep-2024', 'Oct-2024', 'Nov-2024', 'Dec-2024']].notnull().any(axis = 1)
fileTracker['Jan_Jun_2025'] = powerBITable[['Jan-2025', 'Feb-2025', 'Mar-2025', 'Apr-2025', 'May-2025', 'Jun-2025']].notnull().any(axis = 1)
fileTracker['Jul_Dec_2025'] = powerBITable[['Jul-2025', 'Aug-2025', 'Sep-2025', 'Oct-2025', 'Nov-2025', 'Dec-2025']].notnull().any(axis = 1)
fileTracker['Jan_Jun_2026'] = powerBITable[['Jan-2026', 'Feb-2026', 'Mar-2026', 'Apr-2026', 'May-2026', 'Jun-2026']].notnull().any(axis = 1)

fileTracker

In [ ]:
fileTracker.shape

## Export the Usage Information File

In [ ]:
folderPath = r"C:\Users\aharrima\OneDrive - University of Tennessee\Documents\Usage Database Update\Usage File Tracker Reports" # Change this path for your computer as needed.

directory = os.chdir(folderPath)

# Export Data
fileTrackerExportName = r''+fileType+' Usage File Tracker Report.xlsx'

fileTracker.to_excel(fileTrackerExportName, index = False)

### Close the Connection to the Database

In [ ]:
conn.close()

# Next Steps:

### Looking to update the PowerBI report?

Go to your locally saved copy of the report and refresh any tables (both the main table and Long table, if both are in use) that need to be updated, then re-upload the PowerBI file to the server. You should not need to re-connect to SQLite3.

### Looking to query the database for usage?

You're ready to go! Open the database in SQLite3 or connect to it in another code file to access the new usage.

### Looking to update another file type?

Change the file type and folder paths, then run the process again. 

If you just finished and/or are looking to run a large file type (such as Title or Item reports) you may want to restart the Kernel to free up memory. If you do this, start from the package loading cell.

#### NOTE: once you restart the kernel or run cells again, any information saved in variables will be lost!

### Just finished making changes to the code?

Make sure to re-upload it to GitHub (make sure to use a pull request instead of a standard upload) to save your changes.